In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, joblib
from sklearn.ensemble import IsolationForest

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.features import to_matrix, ClippedScaler
from src.evaluation import evaluate
from src.pipeline import DATA, RESULTS

SEED = 11
cols = joblib.load(RESULTS / "scaler_v2.joblib")["cols"]
u = pd.read_parquet(DATA / "unsw.parquet", columns=cols + ["row_id", "Attack"])
n = len(u)
c1, c2, gap = int(0.6 * n), int(0.7 * n), 50_000
is_b = (u["Attack"] == "Benign").to_numpy()

tr = u.iloc[:c1][is_b[:c1]].sample(200_000, random_state=SEED)
va = u.iloc[c1:c2][is_b[c1:c2]]
te = u.iloc[c2 + gap:]
print("test segment attack counts:")
print(te["Attack"].value_counts())

x_tr = to_matrix(tr, cols)
sc = ClippedScaler(clip=5.0).fit(x_tr)
model = IsolationForest(n_estimators=200, random_state=SEED, n_jobs=-1).fit(sc.transform(x_tr))

def score(df, split):
    s = -model.score_samples(sc.transform(to_matrix(df, cols)))
    return pd.DataFrame({"id": df["row_id"].to_numpy(), "score": s,
                         "label": (df["Attack"] != "Benign").astype(int).to_numpy(),
                         "attack_type": df["Attack"].astype(str).to_numpy(), "split": split})

val_s = score(va.sample(min(len(va), 200_000), random_state=SEED), "val")
te_b = te[te["Attack"] == "Benign"].sample(200_000, random_state=SEED)
test_s = score(pd.concat([te_b, te[te["Attack"] != "Benign"]]), "test")

r, o = evaluate(test_s, val_df=val_s), evaluate(test_s)
print({k: round(r[k], 3) for k in ["auroc", "tpr@fpr0.01", "real_fpr@0.01", "macro_tpr@fpr0.05", "real_fpr@0.05"]})
print("oracle macro TPR@5%:", round(o["macro_tpr@fpr0.05"], 3))
print({k: round(v, 3) for k, v in r["per_attack_tpr@fpr0.01"].items()})

test segment attack counts:
Attack
Benign            620943
Exploits           16052
Fuzzers            10613
Generic             7102
Reconnaissance      6591
DoS                 2894
Analysis            1136
Backdoor             938
Shellcode            728
Worms                 86
Name: count, dtype: int64[pyarrow]
{'auroc': 0.943, 'tpr@fpr0.01': 0.0, 'real_fpr@0.01': 0.006, 'macro_tpr@fpr0.05': 0.594, 'real_fpr@0.05': 0.045}
oracle macro TPR@5%: 0.627
{'Analysis': 0.0, 'Backdoor': 0.0, 'DoS': 0.002, 'Exploits': 0.0, 'Fuzzers': 0.001, 'Generic': 0.0, 'Reconnaissance': 0.0, 'Shellcode': 0.0, 'Worms': 0.0}
